In [ ]:
# Environment and imports
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from functools import partial

from spikingjelly.activation_based import neuron, surrogate, functional, layer

from src.ablation_cam import FractionalAblationCAM
from src.sam import SpikeActivationMap
from src.snn_gradcam import SNNGradCAM
from src.utils.config_utils import setup_config_dictionary
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities, plot_aggregated_cam

# Repro + device
print(torch.seed())
# torch.manual_seed(23296697184421296 58)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = setup_config_dictionary()
config["DATASET_NAME"] = "dvs_gesture"
config["MODEL_NAME"] = "simple_snn"
config["NORM_TYPE"] = "none"
config["TAU"] = 2.0

print('Device:', device)
print('Dataset:', config['DATASET_NAME'])
print('Model:', config['MODEL_NAME'])

In [ ]:
train_loader, test_loader = get_dataloaders(
    config['DATASET_NAME'],
    batch_size=1,
)

sample_events, sample_labels = next(iter(test_loader))
sample_events = sample_events.transpose(0, 1)

T, B, C, H, W = sample_events.shape
print('Events Shape (T, B, C, H, W):', (T, B, C, H, W))

model = get_model(
    config["MODEL_NAME"],
    num_classes=len(train_loader.dataset.classes),
    in_channels=C,
    spiking_neuron=neuron.LIFNode,
    pretrained=True,
    tau=config["TAU"],
    detach_reset=False,
    surrogate_function=surrogate.ATan()
)
functional.set_step_mode(model, 'm')
functional.set_backend(model, "torch", instance=neuron.LIFNode)

model.to(device)

interactive_load_model(model, config['MODEL_SAVE_DIR'], config['DATASET_NAME'])
model.eval()
print("Model loading!")

In [ ]:
sample_events = sample_events.to(device)
sample_label = sample_labels.to(device)
print('Label:', int(sample_label[0].item()))

outputs = model(sample_events)  # (T, B, num_classes)
time_mean = outputs.mean(dim=0)
pred = time_mean.argmax(dim=1)
print(f'Predicted class: {int(pred[0].item())} with confidence {F.softmax(time_mean, dim=1)[0,pred].item():.4f}')

conv_layers = [m for m in model.modules() if isinstance(m, layer.Conv2d)]
lif_layers = [m for m in model.modules() if isinstance(m, neuron.LIFNode)]
print(f'Found {len(conv_layers)} conv layers and {len(lif_layers)} LIF layers.')

In [ ]:
def build_model_copy():
    m = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        in_channels=C,
        spiking_neuron=neuron.LIFNode,
        pretrained=False,  # copy weights from the loaded model
        tau=config["TAU"],
        detach_reset=False,
        surrogate_function=surrogate.ATan(),
    )
    functional.set_step_mode(m, 'm')
    functional.set_backend(m, "torch", instance=neuron.LIFNode)
    m.load_state_dict(model.state_dict(), strict=True)
    m.to(device)
    m.eval()
    return m

In [ ]:
sample_frames = sample_events[:, 0].detach().cpu()  # (T, C, H, W)
if sample_frames.shape[1] == 2:  # (T, 2, H, W) -> (T, 1, H, W)
    combined = combine_polarities(sample_frames.permute(0,2,3,1).numpy())  # (T,H,W)
    combined = np.expand_dims(combined, axis=1)
else: # 3 channels, assume RGB
    combined = sample_frames.numpy()
    
for layer_index in range(len(conv_layers)):
    print(f'Targeting layer index {layer_index} for CAMs, out of {len(conv_layers)} conv layers and {len(lif_layers)} LIF layers.')
    # Build three independent model copies
    model_copies = [build_model_copy() for _ in range(3)]

    # 1) Grad-CAM
    gc_model = model_copies[2]
    for p in gc_model.parameters():
        p.requires_grad_(True)

    functional.reset_net(gc_model)

    gc_conv_layers = [m for m in gc_model.modules() if isinstance(m, layer.Conv2d)]
    gc_target_layer = gc_conv_layers[layer_index]
    grad_cam = SNNGradCAM(gc_model, gc_target_layer)
    
    gc_model.train()
    for m in gc_model.modules():
        if isinstance(m, (layer.BatchNorm1d, layer.BatchNorm2d, layer.BatchNorm3d, layer.Dropout, layer.Dropout2d)):
            m.eval()

    grad_cam_cams = grad_cam.generate(sample_events, class_idx=pred[0].item())
    grad_cam_cams = grad_cam_cams.squeeze(0)  # Remove batch dim

    # 2) Fractional Ablation CAM
    fa_model = model_copies[0]
    functional.reset_net(fa_model)
    fa_lif_layers = [m for m in fa_model.modules() if isinstance(m, neuron.LIFNode)]
    fa_target_layer = fa_lif_layers[layer_index]
    fractional_ablation_cam = FractionalAblationCAM(fa_model, fa_target_layer, n_steps=8, channel_batch_size=8)

    with torch.no_grad():
        # We must make it time-first
        activations, baseline_outputs = fractional_ablation_cam.extract_activations_outputs(sample_events)
        importance, _, _, _ = fractional_ablation_cam.compute_channel_importance(sample_events, class_idx=pred[0].item())
    ablation_cams = fractional_ablation_cam.compute_cam(sample_events, importance)

    # 3) Spike Activation Map
    sam_model = model_copies[1]
    functional.reset_net(sam_model)
    sam_lif_layers = [m for m in sam_model.modules() if isinstance(m, neuron.LIFNode)]
    sam_target_layer = sam_lif_layers[layer_index]
    spike_activation_map = SpikeActivationMap(sam_model, sam_target_layer)

    with torch.no_grad():
        sam_cams = spike_activation_map.generate(sample_events)
        sam_cams = sam_cams.squeeze(0)  # Remove batch dim
        

    print('Ablation CAMs shape:', ablation_cams.shape)
    print('SAM CAMs shape:', sam_cams.shape)
    print('Grad-CAM CAMs shape:', grad_cam_cams.shape)

    plot_frames(combined)
    print("Ablation CAMs over time:")
    plot_cam_time_series(combined, ablation_cams, per_frame_cam_norm=False)
    print("SAM CAMs over time:")
    plot_cam_time_series(combined, sam_cams, per_frame_cam_norm=False)
    print("Grad-CAM CAMs over time:")
    plot_cam_time_series(combined, grad_cam_cams, per_frame_cam_norm=False)
    print("Yay! This mutes the additional plot.")